### Import Modelling Libraries

In [23]:
# Core
import time
from pathlib import Path
import joblib

# Data
import pandas as pd
import numpy as np

# Models (sklearn + others)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

# External gradient boosting
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Pipeline
from sklearn.pipeline import Pipeline

# Metrics
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Label mapping
LABEL_NAMES = {
    0: "Normal",
    1: "Leak",
    2: "Blockage"
}

# Paths
BASE_PATH   = Path.cwd().parent
SCALED_PATH = BASE_PATH / "data" / "model" / "scaled"
MODEL_PATH  = BASE_PATH / "models"

MODEL_PATH.mkdir(parents=True, exist_ok=True)

print("Imports & config loaded.")


Imports & config loaded.


### Load Scaled Data

In [24]:
# Load data that has been scaled.
# Loading the Robust Scaled data and then, switch to Standard Scaled data later for comparison.

# Load scaled data (ROBUST versions)
X_train = pd.read_csv(SCALED_PATH / "X_train_robust.csv")
X_val   = pd.read_csv(SCALED_PATH / "X_val_robust.csv")
X_test  = pd.read_csv(SCALED_PATH / "X_test_robust.csv")

y_train = pd.read_csv(SCALED_PATH / "y_train.csv").squeeze()
y_val   = pd.read_csv(SCALED_PATH / "y_val.csv").squeeze()
y_test  = pd.read_csv(SCALED_PATH / "y_test.csv").squeeze()

# Verify shapes
print("DATA SHAPES")
print("-" * 30)
print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_val  : {X_val.shape}   | y_val  : {y_val.shape}")
print(f"X_test : {X_test.shape}  | y_test : {y_test.shape}")

# Verify label balance
print("\nLABEL DISTRIBUTION")
print("-" * 30)
print("Train:\n", y_train.value_counts())
print("\nVal:\n", y_val.value_counts())
print("\nTest:\n", y_test.value_counts())

DATA SHAPES
------------------------------
X_train: (23100, 126) | y_train: (23100,)
X_val  : (4200, 126)   | y_val  : (4200,)
X_test : (4200, 126)  | y_test : (4200,)

LABEL DISTRIBUTION
------------------------------
Train:
 label
2    7700
1    7700
0    7700
Name: count, dtype: int64

Val:
 label
2    1400
1    1400
0    1400
Name: count, dtype: int64

Test:
 label
2    1400
1    1400
0    1400
Name: count, dtype: int64


# Define model pipelines

In [25]:
# Pipeline builder
def build_pipeline(name, estimator):
    return Pipeline([
        ("classifier", estimator)
    ])

MODELS = {
    "random_forest": build_pipeline(
        "rf",
        RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            n_jobs=-1,
            random_state=42
        )
    ),

    "xgboost": build_pipeline(
        "xgb",
        XGBClassifier(
            n_estimators = 200,
            lr=0.1
        )
    ),

    "svm": build_pipeline(
        "svm",
        SVC(
            kernel="rbf",
            class_weight = "balanced",
            probability=True
        )
    ),

    "logistic_regression": build_pipeline(
        "lr",
        LogisticRegression(
            max_iter = 1000,
            class_weight="balanced",
            n_jobs=-1
        )
    ),

    "lightgbm": build_pipeline(
        "lgbm",
        LGBMClassifier(
            n_estimators=200,
            n_jobs=-1,
            random_state=42
        )
    ),

    "knn": build_pipeline(
        "knn",
        KNeighborsClassifier(
            n_neighbors=5,
            n_jobs=-1
        )
    )
}

print(f"Defined {len(MODELS)} models:")
for name in MODELS.keys():
    print(f"- {name}")

Defined 6 models:
- random_forest
- xgboost
- svm
- logistic_regression
- lightgbm
- knn


### Train the Models

In [ ]:
# Training the six models
